<a href="https://colab.research.google.com/github/AjayCunanan/prompt-engineering-practice/blob/main/03_self_reflection_summary.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
!pip install -q google-genai

import time, json, re
from google.colab import userdata
from google import genai
from google.genai import types

client = genai.Client(api_key=userdata.get("GEMINI_API_KEY"))
MODEL = "gemini-3.5-flash-lite"

def ask_llm(prompt, temperature=0.3):
    time.sleep(13)  # stay under the free per-minute limit
    config = types.GenerateContentConfig(temperature=temperature)
    response = client.models.generate_content(model=MODEL, contents=prompt, config=config)
    return response.text

def parse_json(text):
    cleaned = re.sub(r"```(?:json)?", "", text).strip()
    return json.loads(cleaned)

print("Ready!", MODEL)

Ready! gemini-3.5-flash-lite


In [6]:
ARTICLE = """
Maplewood's public library launched a "Library of Things" program in March, letting cardholders borrow
items like power drills, sewing machines, telescopes, and camping gear for up to two weeks. The program
started with 120 items funded by a $45,000 state grant. In its first six months, items were checked out
over 3,200 times, and 38% of borrowers were new library cardholders. The most popular items were the
carpet cleaner and the kids' telescope, each with waitlists of several weeks.

Library director Ana Reyes said the goal was to cut costs for residents who need a tool once or twice a
year. A resident survey found that 71% of borrowers said they would otherwise have bought the item.
Not everything went smoothly: 14 items came back damaged, and staff spent more time than expected
cleaning and checking returns. The library now charges a $5 late fee per day for high-demand items and
added a volunteer "repair crew" that meets monthly.

The city council will vote in November on whether to fund the program permanently at $30,000 per year.
Reyes said she hopes to add 60 more items, including accessibility equipment like wheelchairs and
shower chairs.
"""

In [7]:
def summarize(text):
    prompt = f"""Summarize this article in 3-4 sentences.

Article:
{text}
"""
    return ask_llm(prompt, temperature=0.5)


def critique(text, summary):
    prompt = f"""You are a strict editor reviewing a summary. Be honest and specific.

Original article:
{text}

Summary:
{summary}

Score each criterion from 1 to 10:
- accuracy: every fact matches the article, no made-up details
- completeness: covers the main point, key numbers, the challenges, and what happens next
- concision: no filler, 4 sentences max
- clarity: easy for someone who hasn't read the article

Return ONLY valid JSON:
{{"scores": {{"accuracy": n, "completeness": n, "concision": n, "clarity": n}},
  "overall": average score,
  "missing": [important points left out],
  "errors": [anything inaccurate],
  "suggestions": [specific changes to make]}}
"""
    return parse_json(ask_llm(prompt, temperature=0))


def revise(text, summary, review):
    prompt = f"""Rewrite the summary to fix every issue in the critique.

Original article:
{text}

Current summary:
{summary}

Critique:
{json.dumps(review, indent=2)}

Rules: stay accurate to the article, max 4 sentences, return ONLY the new summary.
"""
    return ask_llm(prompt, temperature=0.3)

In [8]:
# 1. First draft
draft = summarize(ARTICLE)
print("DRAFT:\n", draft)

# 2. Critique the draft
review1 = critique(ARTICLE, draft)
print("\nCRITIQUE:\n", json.dumps(review1, indent=2))

# 3. Revise using the critique
improved = revise(ARTICLE, draft, review1)
print("\nIMPROVED:\n", improved)

# 4. Score the new version to see if it got better
review2 = critique(ARTICLE, improved)
print("\nNEW SCORE:", review2["overall"], "| Missing:", review2["missing"])

DRAFT:
 Maplewood's public library launched a successful "Library of Things" program, allowing cardholders to borrow non-traditional items like tools and camping gear. Funded by a state grant, the initiative attracted many new users and saved residents money, though it also created challenges with damaged goods and maintenance. The city council will vote in November on whether to permanently fund the program, which the library hopes to expand with accessibility equipment.

CRITIQUE:
 {
  "scores": {
    "accuracy": 10,
    "completeness": 7,
    "concision": 10,
    "clarity": 10
  },
  "overall": 9.25,
  "missing": [
    "Specific usage statistics (3,200 checkouts, 38% new cardholders, or the 71% who avoided purchases)",
    "Specific administrative changes made to address challenges ($5 late fee and volunteer repair crew)"
  ],
  "errors": [],
  "suggestions": [
    "Incorporate a few key data points, such as the 3,200 checkouts or the upcoming $30,000 funding vote amount, to improve

In [9]:
print(f"BEFORE (score {review1['overall']}):\n{draft}\n")
print(f"AFTER  (score {review2['overall']}):\n{improved}")

BEFORE (score 9.25):
Maplewood's public library launched a successful "Library of Things" program, allowing cardholders to borrow non-traditional items like tools and camping gear. Funded by a state grant, the initiative attracted many new users and saved residents money, though it also created challenges with damaged goods and maintenance. The city council will vote in November on whether to permanently fund the program, which the library hopes to expand with accessibility equipment.

AFTER  (score 9.75):
Maplewood's public library successfully launched a "Library of Things" funded by a state grant, recording over 3,200 checkouts in six months and attracting 38% new cardholders. While the program saved residents money—with 71% avoiding retail purchases—it also created maintenance challenges that prompted the library to institute a $5 daily late fee for high-demand items and form a volunteer repair crew. The city council will vote in November on whether to permanently fund the program 